In [ ]:
import pandas as pd, glob

WL, PCT = "spike", 80          # change WL to "ramp" for the second workload
s = pd.read_csv(f"ramp_static.csv")
d = pd.read_csv(f"ramp_dynamic_80.csv")

def kpis(df):
    return {
        "Avg CPU util (%)":       round(df.cpu_util.mean() * 100, 1),
        "Avg memory util (%)":    round(df.mem_util.mean() * 100, 1),
        "Avg response time (s)":  round(df.resp_time[df.resp_time > 0].mean(), 2),
        "Avg throughput (req/s)": round(df.throughput.mean(), 2),
        "SLA violations (total)": int(df.sla_violations.sum()),
        "VM migrations":          int(df.migrations.max()),
        "Avg active hosts":       round(df.active_hosts.mean(), 2),
        "Peak saturated hosts":   int(df.saturated_hosts.max()),
    }
pd.DataFrame({"Static": kpis(s), "Dynamic": kpis(d)})

,Static,Dynamic
Avg CPU util (%),42.50,43.00
Avg memory util (%),16.40,16.60
Avg response time (s),0.88,0.88
Avg throughput (req/s),12.76,12.76
SLA violations (total),68.00,68.00
VM migrations,0.00,123.00
Avg active hosts,6.00,5.94
Peak saturated hosts,0.00,0.00


In [ ]:
m = pd.read_csv(f"ramp_migrations_80.csv")
print("total migrations:", len(m))
print("\nper VM:\n", m.groupby("vm").size().sort_values(ascending=False).to_string())

# ping-pong: a VM moves back to the host it just left
m = m.sort_values(["vm", "time_completed"])
m["prev_from"] = m.groupby("vm")["from_host"].shift(0)
m["prev_to"]   = m.groupby("vm")["to_host"].shift(1)
m["prev_time"] = m.groupby("vm")["time_completed"].shift(1)
pp = m[(m.to_host == m.from_host.shift(1)) & (m.vm == m.vm.shift(1))]
print("\nping-pong moves (VM returns to the host it just left):", len(pp))

# migrations per workload phase
bins = [0, 150, 300, 450, 700] if WL == "spike" else [0, 100, 350, 450, 700]
m["phase"] = pd.cut(m.time_completed, bins=bins)
print("\nmigrations per phase:\n", m.groupby("phase", observed=True).size().to_string())

window
(0, 100]      22
(100, 350]    53
(350, 450]     1
(450, 700]    39
total migrations: 115

per VM:
 vm
0     14
4     11
7     11
6     10
10    10
11    10
1      9
5      9
9      9
3      8
2      7
8      7
window
(0, 100]      22
(100, 350]    53
(350, 450]     1
(450, 700]    39

ping-pong moves (VM returns to the host it just left): 21

migrations per phase:
 phase
(0, 150]      34
(150, 300]    37
(300, 450]     5
(450, 700]    39


In [ ]:
     # make sure WL is "ramp" here

m = pd.read_csv(f"ramp_migrations_80.csv")
m["window"] = pd.cut(m.time_completed, bins=[0, 100, 350, 450, 700])
print(m.groupby("window", observed=True).size().to_string())
print("total migrations:", len(m))
print("per VM:\n", m.groupby("vm").size().sort_values(ascending=False).to_string())

m = m.sort_values(["vm", "time_completed"])
m["window"] = pd.cut(m.time_completed, bins=[0, 100, 350, 450, 700])
print(m.groupby("window", observed=True).size().to_string())
pp = m[(m.to_host == m.from_host.shift(1)) & (m.vm == m.vm.shift(1))]
print("ping-pong moves:", len(pp))

bins = [0, 150, 300, 450, 700] if WL == "spike" else [0, 100, 350, 450, 700]
m["window"] = pd.cut(m.time_completed, bins=bins)
print("migrations per window:\n", m.groupby("window", observed=True).size().to_string())

window
(0, 100]      22
(100, 350]    53
(350, 450]     1
(450, 700]    39
total migrations: 115
per VM:
 vm
0     14
4     11
7     11
6     10
10    10
11    10
1      9
5      9
9      9
3      8
2      7
8      7
window
(0, 100]      22
(100, 350]    53
(350, 450]     1
(450, 700]    39
ping-pong moves: 21
migrations per window:
 window
(0, 150]      34
(150, 300]    37
(300, 450]     5
(450, 700]    39
